In [11]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import ast

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)
""
print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [12]:
portfolio = pd.read_csv('../dataset/portfolio.csv') # 프로모션 제공에 대한 정보 
profile = pd.read_csv('../dataset/profile.csv') # 고객들에 대한 인구통계학적 정보 
menu = pd.read_csv('../dataset/starbucks_menu_260112.csv')
transcript = pd.read_csv('../dataset/transcript.csv') # 각 고객들이 받은 프로모션에 대한 로그 

## portfolio 전처리

In [13]:
portfolio.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
portfolio.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   reward      10 non-null     int64
 1   channels    10 non-null     str  
 2   difficulty  10 non-null     int64
 3   duration    10 non-null     int64
 4   offer_type  10 non-null     str  
 5   id          10 non-null     str  
dtypes: int64(3), str(3)
memory usage: 612.0 bytes


In [14]:
portfolio.head()

,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [15]:
portfolio['channels'] = portfolio['channels'].apply(ast.literal_eval) # 문자열 텍스트를 파이썬 리스트 형태로 변환

channel_list = ['web', 'email', 'mobile', 'social']	

# 채널별 사용 유무 컬럼 & 활용된 채널 총 개수 컬럼 생성하는 함수
def channel_encoding(df):
    for ch in channel_list:
        df[ch] = df['channels'].apply(lambda x: 1 if ch in x else 0)
    df['channel_count'] = df[channel_list].sum(axis=1) # 활용된 채널 총 개수

channel_encoding(portfolio)

portfolio.drop(columns='channels', inplace=True, errors=False) # channels 컬럼 제거 

In [16]:
portfolio.sort_values(by='offer_type')

,reward,difficulty,duration,offer_type,id,web,email,mobile,social,channel_count
0,10,10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1,3
1,10,10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1,4
3,5,5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0,3
8,5,5,5,bogo,f19421c1d4aa40978ebb69ca19b0e20d,1,1,1,1,4
4,5,20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0,2
5,3,7,7,discount,2298d6c36e964ae4a3e7e9706d1fb8c2,1,1,1,1,4
6,2,10,10,discount,fafdcd668e3743c1bb461111dcafc2a4,1,1,1,1,4
9,2,10,7,discount,2906b810c7d4411798c6938adc9daaa5,1,1,1,0,3
2,0,0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0,3
7,0,0,3,informational,5a8bc65990b245e5a138643cd4eb9837,0,1,1,1,3


## profile 전처리

In [17]:
profile.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               17000 non-null  int64  
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 664.2 KB


In [18]:
profile.head()

,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [19]:
profile.isna().sum()

gender              2175
age                    0
id                     0
became_member_on       0
income              2175
dtype: int64

In [20]:
profile['gender'].value_counts()

gender
M    8484
F    6129
O     212
Name: count, dtype: int64

### profile['gender'] == 'O'인 경우 ---> 미기입은 아님

In [21]:
# profile['gender'] = profile['gender'].where(profile['gender']!='O', other=None)
# profile['gender'].value_counts()

### gender열이 NaN인 행과 income열이 NaN인 행이 정확히 일치함 --> 나머지는 정상, gender과 income의 결측치가 함께 존재함
### 해당 이상 행의 age 열 값도 모두 118인 것으로 보아 이상치 --> 결측치 처리

In [22]:
# gender 열과 income 열의 결측치 행 확인
profile[(profile['gender'].isna()) & profile['income'].isna()]['age'].value_counts() # 2175 

age
118    2175
Name: count, dtype: int64

In [23]:
# age 열 값이 118인 행 모두 결측치 처리
profile['age'] = profile['age'].where(profile['age']!=118, other=None) 

In [24]:
profile[profile['age']>=80]

,gender,age,id,became_member_on,income
33,F,96.0,868317b9be554cb18e50bc68484749a2,20171117,89000.0
94,F,89.0,4264b1d027cc493281bba4f44bfedaca,20171114,87000.0
98,F,90.0,1c587de019994f93a242c6864fd7bc55,20151210,98000.0
126,M,83.0,4c29d22467af4d7faa137c4eedd65340,20180127,46000.0
131,F,89.0,3dae0eadb47149b0b9b548d14548524b,20180114,65000.0
...,...,...,...,...,...
16914,M,87.0,d1c4500ace2e45e9a45d3cd2fccac8d8,20140920,59000.0
16933,M,85.0,a65353ea28ff442aabfb39eb974326e3,20161001,96000.0
16938,F,89.0,da7bf9d84fd74a72bdee595007bcca7a,20170413,68000.0
16981,M,84.0,1966fa40d2f84620b2b1b9b64f8e0209,20160629,93000.0


### age 값이 100 이상인 행 처리? 
--> 지울거면 명확한 기준이 있어야함

--> 기준이 80 이상이 될 수도 있고... 그 기준 필요!!!

--> 그 중 118인 경우는 gender & income 컬럼이 결측값이기 때문에 이상치로 처리할 명확한 기준 존재

In [25]:
profile['age'].isna().sum()

np.int64(2175)

In [26]:
profile['gender'].isna().sum()

np.int64(2175)

In [27]:
# 나이 & 성별 통합 범주화 함수
def group_age_gender(df):
    if pd.isna(df['gender']):   # gender 결측치인 경우
        return '미기입'
    elif df['gender']=='O':
        return 'Others'
    elif df['gender']=='M':     # 남성
        if df['age']<20:
            return '20세 미만 남성'
        elif df['age']<30:
            return '20대 남성'
        elif df['age']<40:
            return '30대 남성'
        elif df['age']<50:
            return '40대 남성'
        elif df['age']<60:
            return '50대 남성'
        elif df['age']<70:
            return '60대 남성'
        else:
            return '60+ 남성'
    else:                        # 여성
        if df['age']<20:
            return '20세 미만 여성'
        elif df['age']<30:
            return '20대 여성'
        elif df['age']<40:
            return '30대 여성'
        elif df['age']<50:
            return '40대 여성'
        elif df['age']<60:
            return '50대 여성'
        elif df['age']<70:
            return '60대 여성'
        else:
            return '60+ 여성'

profile['age_gender'] = profile.apply(group_age_gender, axis=1)
profile['age_gender'].value_counts()

age_gender
미기입          2175
50대 남성       1925
60대 남성       1602
50대 여성       1560
60+ 여성       1451
40대 남성       1434
60+ 남성       1395
60대 여성       1350
30대 남성       1008
20대 남성        960
40대 여성        835
30대 여성        495
20대 여성        393
Others        212
20세 미만 남성     160
20세 미만 여성      45
Name: count, dtype: int64

In [28]:
profile.head()

,gender,age,id,became_member_on,income,age_gender
0,NaN,NaN,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN,미기입
1,F,55.0,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0,50대 여성
2,NaN,NaN,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN,미기입
3,F,75.0,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0,60+ 여성
4,NaN,NaN,a03223e636434f42ac4c3df47e8bac43,20170804,NaN,미기입


## menu 전처리

In [29]:
menu.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
menu.info()

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   제품코드          195 non-null    int64
 1   제품명           195 non-null    str  
 2   1회 제공량(kcal)  195 non-null    str  
 3   포화지방(g)       195 non-null    str  
 4   단백질(g)        195 non-null    str  
 5   지방(g)         195 non-null    str  
 6   트랜스지방(g)      195 non-null    str  
 7   나트륨(mg)       195 non-null    str  
 8   당류(g)         195 non-null    str  
 9   카페인(mg)       195 non-null    str  
 10  콜레스테롤(mg)     195 non-null    str  
 11  탄수화물(g)       195 non-null    str  
dtypes: int64(1), str(11)
memory usage: 18.4 KB


## transcript 전처리

In [30]:
transcript.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
transcript.info()

<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   person  306534 non-null  str  
 1   event   306534 non-null  str  
 2   value   306534 non-null  str  
 3   time    306534 non-null  int64
dtypes: int64(1), str(3)
memory usage: 9.4 MB


In [31]:
transcript['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

In [32]:
# value 컬럼 key 확인

transcript['value'] = transcript['value'].apply(ast.literal_eval) # 문자열 텍스트를 파이썬 딕셔너리 형태로 변환

all_keys = set() # 집합 생성 (중복 알아서 걸러줌)

for val in transcript['value']: # val은 딕셔너리
    all_keys.update(val.keys()) # 딕셔너리의 키만 뽑아서 집합에 추가

print(all_keys) # {'reward', 'offer_id', 'amount', 'offer id'}

{'offer_id', 'offer id', 'amount', 'reward'}


In [33]:
# value 딕셔너리 안에 'amount'라는 키가 있는지 검사해서 True/False 리스트(마스크) 생성
has_amount = transcript['value'].apply(lambda x: 'amount' in x)

transcript[has_amount]['event'].value_counts()
# transaction    138953

event
transaction    138953
Name: count, dtype: int64

In [34]:
# value 딕셔너리 안에 'reward'라는 키가 있는지 검사해서 True/False 리스트(마스크) 생성
has_reward = transcript['value'].apply(lambda x: 'reward' in x)

transcript[has_reward]['event'].value_counts()
# offer completed    33579

event
offer completed    33579
Name: count, dtype: int64

In [35]:
# value 딕셔너리 안에 'offer_id'라는 키가 있는지 검사해서 True/False 리스트(마스크) 생성
has_offer_id = transcript['value'].apply(lambda x: 'offer_id' in x)

transcript[has_offer_id]['event'].value_counts()

event
offer completed    33579
Name: count, dtype: int64

In [36]:
completed = transcript[transcript['event']=='offer completed']
display(completed)

,person,event,value,time
12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,offer completed,{'offer_id': '2906b810c7d4411798c6938adc9daaa5...,0
12672,fe97aa22dd3e48c8b143116a8403dd52,offer completed,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...,0
12679,629fc02d56414d91bca360decdfa9288,offer completed,{'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...,0
12692,676506bad68e4161b9bbaffeb039626b,offer completed,{'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...,0
12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,offer completed,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...,0
...,...,...,...,...
306475,0c027f5f34dd4b9eba0a25785c611273,offer completed,{'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...,714
306497,a6f84f4e976f44508c358cc9aba6d2b3,offer completed,{'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...,714
306506,b895c57e8cd047a8872ce02aa54759d6,offer completed,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...,714
306509,8431c16f8e1d440880db371a68f82dd0,offer completed,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...,714


### event == 'offer completed'인 행에 대해선 Value 딕셔너리에 offer_id & reward 존재 

- 거래(transaction) : 모두 amount
- 프로모션 받음(offer received) : 모두 offer id
- 프로모션 확인(offer viewed) : 모두 offer id
- 프로모션 완료(offer completed) : 모두 offer_id & reward

### transcript의 reward가 portfolio의 reward와 정확히 같은 값인데 
### transcript의 reward는 offer completed에만 값이 채워져있고
### portfolio의 reward는 offer에 해당하는 event 모두에 값이 채워져있다는 차이점!
### portfolio의 reward만 남기고 transcript의 reward는 지워도 될지? --> 지우자! (필요하면 나중에 portfolio에서 컬럼 다시 생성할 수 있음)

In [37]:
# value 딕셔너리 안에 'offer id'라는 키가 있는지 검사해서 True/False 리스트(마스크) 생성
has_offer_id2 = transcript['value'].apply(lambda x: 'offer id' in x)

transcript[has_offer_id2]['event'].value_counts()

event
offer received    76277
offer viewed      57725
Name: count, dtype: int64

In [38]:
transcript['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

In [39]:
# value 딕셔너리 내용으로 각 컬럼 생성하는 함수 
def parse_value_dict(df):
    
    df['amount'] = df['value'].str.get('amount') # 'amount' 키 값만 뽑아옴 (없으면 NaN)
    # df['reward'] = df['value'].str.get('reward') # 'reward' 컬럼 제거

    # offer_id 컬럼 생성
    temp_id1 = df['value'].str.get('offer id') # 띄어쓰기 버전
    temp_id2 = df['value'].str.get('offer_id') # 언더바 버전
    df['offer_id'] = temp_id1.fillna(temp_id2) # 두 버전 모두 결측값인 경우 (transaction) 결측값으로 유지

    df.drop(columns='value', inplace=True, errors=False)
    
    return df

transcript = parse_value_dict(transcript)
transcript.head()

,person,event,time,amount,offer_id
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0


In [40]:
transcript['time'].describe()

count    306534.000000
mean        366.382940
std         200.326314
min           0.000000
25%         186.000000
50%         408.000000
75%         528.000000
max         714.000000
Name: time, dtype: float64

#### 코호트 분석에서는 1.5일같은 애매한 값은 '버림' 하여 1일로 간주

In [41]:
# transcript['time_days'] = transcript['time'] // 24 # 24로 나눈 몫 --> 버림의 의미와 동일
# transcript.head()

In [42]:
def making_days(df): 
    df['time_days'] = df['time'] // 24 # 24로 나눈 몫 --> 버림의 의미와 동일
    return df

making_days(transcript)
transcript.head()

,person,event,time,amount,offer_id,time_days
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0


In [43]:
transcript.info()

<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   person     306534 non-null  str    
 1   event      306534 non-null  str    
 2   time       306534 non-null  int64  
 3   amount     138953 non-null  float64
 4   offer_id   167581 non-null  object 
 5   time_days  306534 non-null  int64  
dtypes: float64(1), int64(2), object(1), str(2)
memory usage: 14.0+ MB


## 전처리 완료된 세 테이블을 통합 (merge)

In [44]:
transcript.info()

<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   person     306534 non-null  str    
 1   event      306534 non-null  str    
 2   time       306534 non-null  int64  
 3   amount     138953 non-null  float64
 4   offer_id   167581 non-null  object 
 5   time_days  306534 non-null  int64  
dtypes: float64(1), int64(2), object(1), str(2)
memory usage: 14.0+ MB


In [45]:
profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               14825 non-null  float64
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
 5   age_gender        17000 non-null  str    
dtypes: float64(2), int64(1), str(3)
memory usage: 797.0 KB


In [46]:
portfolio.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   reward         10 non-null     int64
 1   difficulty     10 non-null     int64
 2   duration       10 non-null     int64
 3   offer_type     10 non-null     str  
 4   id             10 non-null     str  
 5   web            10 non-null     int64
 6   email          10 non-null     int64
 7   mobile         10 non-null     int64
 8   social         10 non-null     int64
 9   channel_count  10 non-null     int64
dtypes: int64(8), str(2)
memory usage: 932.0 bytes


In [47]:
merged_df = pd.merge(transcript, profile, left_on='person', right_on='id', how='left')
merged_df.drop(columns='id', inplace=True, errors=False)
merged_df

,person,event,time,amount,offer_id,time_days,gender,age,became_member_on,income,age_gender
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,F,75.0,20170509,100000.0,60+ 여성
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,NaN,NaN,20170804,NaN,미기입
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0,M,68.0,20180426,70000.0,60대 남성
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0,NaN,NaN,20170925,NaN,미기입
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0,NaN,NaN,20171002,NaN,미기입
...,...,...,...,...,...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,transaction,714,1.59,None,29,M,66.0,20180101,47000.0,60대 남성
306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,9.53,None,29,M,52.0,20180408,62000.0,50대 남성
306531,a00058cf10334a308c68e7631c529907,transaction,714,3.61,None,29,F,63.0,20130922,52000.0,60대 여성
306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,3.53,None,29,M,57.0,20160709,40000.0,50대 남성


In [48]:
merged_df = pd.merge(merged_df, portfolio, left_on='offer_id', right_on='id', how='left')
merged_df.drop(columns='id', inplace=True, errors=False)
merged_df

,person,event,time,amount,offer_id,time_days,gender,age,became_member_on,income,age_gender,reward,difficulty,duration,offer_type,web,email,mobile,social,channel_count
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,F,75.0,20170509,100000.0,60+ 여성,5.0,5.0,7.0,bogo,1.0,1.0,1.0,0.0,3.0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,NaN,NaN,20170804,NaN,미기입,5.0,20.0,10.0,discount,1.0,1.0,0.0,0.0,2.0
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0,M,68.0,20180426,70000.0,60대 남성,2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0,3.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0,NaN,NaN,20170925,NaN,미기입,2.0,10.0,10.0,discount,1.0,1.0,1.0,1.0,4.0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0,NaN,NaN,20171002,NaN,미기입,10.0,10.0,5.0,bogo,1.0,1.0,1.0,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,transaction,714,1.59,None,29,M,66.0,20180101,47000.0,60대 남성,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,9.53,None,29,M,52.0,20180408,62000.0,50대 남성,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306531,a00058cf10334a308c68e7631c529907,transaction,714,3.61,None,29,F,63.0,20130922,52000.0,60대 여성,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,3.53,None,29,M,57.0,20160709,40000.0,50대 남성,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


transaction이 아닌 건에 대해서는 (offer event 모두 포함) reward_y (portfolio) 는 None 이 아님. --> 138953건

reward_x (transcript) 는 offer_completed인 건에 대해서만 None이 아님. --> 33579건 

In [49]:
# received  = full[full['event'] == 'offer received' ][['person','offer_id','time']].rename(columns={'time':'t_received'})
# viewed    = full[full['event'] == 'offer viewed'   ][['person','offer_id','time']].rename(columns={'time':'t_viewed'})
# completed = full[full['event'] == 'offer completed'][['person','offer_id','time']].rename(columns={'time':'t_completed'})
# #중복 제거
# viewed = viewed.sort_values('t_viewed').drop_duplicates(['person','offer_id'], keep='first')
# completed = completed.sort_values('t_completed').drop_duplicates(['person','offer_id'], keep='first')
# received = received.sort_values('t_received').drop_duplicates(['person','offer_id'], keep='first')
# # 병합
# funnel = received.merge(viewed, on=['person','offer_id'], how='left')
# funnel = funnel.merge(completed, on=['person','offer_id'], how='left')
# # 시간 간격 계산 (시간 단위)
# funnel['gap_to_view']     = funnel['t_viewed']    - funnel['t_received']   # 수신 → 열람
# funnel['gap_to_complete'] = funnel['t_completed'] - funnel['t_received']   # 수신 → 완료
# funnel['gap_view_to_complete'] = funnel['t_completed'] - funnel['t_viewed'] # 열람 → 완료
# # 일(day) 단위로도 생성
# funnel['gap_to_view_day']     = funnel['gap_to_view']     / 24
# funnel['gap_to_complete_day'] = funnel['gap_to_complete'] / 24